# 資産運用向け Snowflake AI ハンズオン## Part 1: AI Functions で非構造化データを構造化する決算カンファレンスコールの PDF と企業ニュースを **Cortex AI Functions** で構造化し、Marketplace 由来の市場データと突合できる状態にします。

### このパートで体験できること| AI Function | 用途 | 資産運用での活用例 ||---|---|---|| `AI_PARSE_DOCUMENT` | PDF を Markdown に変換 | 決算コール16ページを SQL で扱える形に || `SPLIT_TEXT_MARKDOWN_HEADER` | 見出し単位のチャンク化 | 発言者ごとに分割して検索できる形に || `AI_EXTRACT` | 事実の構造化抽出 | ガイダンス・セグメント売上を JSON 抽出 || `AI_SENTIMENT` | 感情分析 | ニュースが強気か弱気かを自動判定 || `AI_CLASSIFY` | テキスト分類 | 記事をイベント種別に分類 || `AI_AGG` | 複数行の横断要約 | 銘柄別にニュース全体を要約 |

### 体験ポイント> **「16ページの決算コールが、発言者ごとに検索できるテーブルに。」**>> PDF をステージに置いたまま、SQL だけで構造化できます。> ダウンロードも Python ライブラリの導入も不要です。

### 前提条件- `setup.sql` 実行済み（DB・スキーマ・ステージ・キュレーション済テーブルの作成完了）- ウェアハウスに `SNOW_AM_WH` を選択していること> ⏱️ **このパートの目安時間: 35分**

In [ ]:
-- 環境設定USE DATABASE SNOW_AM_DB;USE SCHEMA MARKET_INTELLIGENCE;USE WAREHOUSE SNOW_AM_WH;SELECT CURRENT_ROLE() AS "ロール",       CURRENT_REGION() AS "リージョン",       CURRENT_DATABASE() AS "データベース",       CURRENT_SCHEMA() AS "スキーマ",       CURRENT_WAREHOUSE() AS "ウェアハウス";

## 1. 非構造化データを構造化する運用部門が読みたいのは決算カンファレンスコールですが、これは16ページの PDF です。このままでは SQL で分析できません。Snowflake では **PDF をステージに置いたまま**、SQL 関数だけで解析・構造化できます。ファイルをどこかにダウンロードしたり、Python の PDF ライブラリを入れたりする必要はありません。### 1-1. ステージ上の決算コール PDF を確認する`setup.sql` の Step 4 で、GitHub リポジトリから `DOC_STAGE` にファイルを搬入済みです。`DIRECTORY()` 関数でステージの中身をテーブルのように SELECT できます。

In [ ]:
-- ステージ上のファイル一覧SELECT RELATIVE_PATH AS "ファイルパス",       SIZE AS "サイズ（バイト）",       LAST_MODIFIED AS "最終更新",       FILE_URL AS "ファイルURL"FROM DIRECTORY(@DOC_STAGE)ORDER BY RELATIVE_PATH;

### AI_PARSE_DOCUMENT とは`AI_PARSE_DOCUMENT` は PDF・画像・Office ファイルを **Markdown テキストに変換**する関数です。| モード | 用途 ||---|---|| `OCR` | テキストの抽出だけを行う。高速 || `LAYOUT` | 見出し・表・段落構造を Markdown として保持する。RAG 用途に推奨 |今回は決算コールの**発言者ごとの構造**を保持したいので `LAYOUT` モードを使います。まずは1ファイルだけを対象に実行し、何が返ってくるかを確認してみましょう。

In [ ]:
-- AI_PARSE_DOCUMENT の単体スモークテスト-- 返り値は OBJECT 型です。まずどんなキーが入っているかを確認します。SELECT    TYPEOF(parsed) AS "返り値の型",    OBJECT_KEYS(parsed) AS "含まれるキー",    LENGTH(parsed:content::VARCHAR) AS "本文の文字数",    LEFT(parsed:content::VARCHAR, 600) AS "本文の冒頭600字"FROM (    SELECT AI_PARSE_DOCUMENT(               TO_FILE('@DOC_STAGE', 'earnings_calls/nvda_q1_fy2027_earnings_call.pdf'),               {'mode': 'LAYOUT', 'page_split': FALSE}           ) AS parsed);

> **💡 ここで確認したいこと**>> - 返り値は `OBJECT` 型で、`content` キーに Markdown テキストが入っています> - 本文の冒頭に `# NVIDIA Corp. (NVDA)` のような **Markdown 見出し**が付いています> - この見出しが後のチャンク化で重要な役割を果たします次に、ステージ上のすべての決算コール PDF をまとめて解析し、テーブルに格納します。`DIRECTORY()` の結果に対して `AI_PARSE_DOCUMENT` を適用すれば、ファイルが増えても同じ SQL で処理できます。

In [ ]:
-- 決算コール PDF を一括解析してテーブル化-- ⚠️ 実行に30秒〜1分程度かかりますCREATE OR REPLACE TABLE RAW_EARNINGS_CALLS ASSELECT    d.RELATIVE_PATH AS FILE_NAME,    d.SIZE AS FILE_SIZE_BYTES,    -- ファイル名から銘柄と決算期のメタデータを付与します    CASE WHEN d.RELATIVE_PATH ILIKE '%nvda%' THEN 'NVDA' END AS TICKER,    CASE WHEN d.RELATIVE_PATH ILIKE '%nvda_q1_fy2027%' THEN DATE '2026-05-20' END AS CALL_DATE,    CASE WHEN d.RELATIVE_PATH ILIKE '%nvda_q1_fy2027%' THEN 'Q1 FY2027' END AS FISCAL_PERIOD_LABEL,    -- 対応する SEC 財務諸表の期末日（後で実績と突合するために持たせます）    CASE WHEN d.RELATIVE_PATH ILIKE '%nvda_q1_fy2027%' THEN DATE '2026-04-26' END AS PERIOD_END_DATE,    CURRENT_TIMESTAMP() AS PARSED_AT,    AI_PARSE_DOCUMENT(        TO_FILE('@DOC_STAGE', d.RELATIVE_PATH),        {'mode': 'LAYOUT', 'page_split': FALSE}    ):content::VARCHAR AS PARSED_MARKDOWNFROM DIRECTORY(@DOC_STAGE) dWHERE d.RELATIVE_PATH ILIKE 'earnings_calls/%.pdf';SELECT TICKER AS "銘柄",       FISCAL_PERIOD_LABEL AS "決算期",       CALL_DATE AS "決算発表日",       LENGTH(PARSED_MARKDOWN) AS "文字数",       FILE_NAME AS "ファイル名"FROM RAW_EARNINGS_CALLS;

### 1-2. 発言者ごとにチャンク化する54,000文字を1件のレコードとして持っていても検索できません。Cortex Search で使えるように**チャンク（断片）に分割**します。ここで先ほどの Markdown 見出しが効いてきます。決算コールの文字起こしでは、**見出しが発言者名になっている**ためです。```# Colette M. Kress      ← CFO の発言# Jen Hsun Huang        ← CEO の発言## Joseph Moore         ← アナリストの質問````SPLIT_TEXT_MARKDOWN_HEADER` を使うと、この見出しを境界にしてチャンク化しつつ、**どの見出しの配下にあったかをメタデータとして保持**できます。結果として「経営陣の発言だけを検索する」といった絞り込みが可能になります。> **注意**: 返り値のオブジェクト構造は `{"chunk": "本文", "headers": {"header_1": "見出し"}}` です。> 見出しは `headers` の下にネストしているため、アクセサは `value:headers:header_1` になります。

### チャンク化の前にページヘッダ・フッターを除去するPDF を解析すると、**全ページに繰り返し入っているヘッダ・フッターがそのまま本文に混ざります**。今回の文字起こしでは、以下が16ページ分繰り返されています。```1-877-FACTSET www.callstreet.comCopyright © 2001-2026 FactSet CallStreet, LLCCorrected Transcript20-May-2026NVIDIA Corp. (NVDA)Q1 2027 Earnings Call```これを残したままチャンク化すると、検索したときに**発言内容ではなくページフッターが上位にヒットしてしまいます**。実際に試すと精度が明確に落ちるため、チャンク化の前に除去します。地味な工程ですが、検索精度への影響は大きいです。

In [ ]:
-- 発言者単位でチャンク化して検索用テーブルを作成CREATE OR REPLACE TABLE GOLD_EARNINGS_CALL_CHUNKS ASWITH cleaned AS (    -- ページヘッダ・フッターと罫線代わりのドット列を除去します    SELECT        TICKER, CALL_DATE, FISCAL_PERIOD_LABEL, FILE_NAME,        REGEXP_REPLACE(            REGEXP_REPLACE(                REGEXP_REPLACE(PARSED_MARKDOWN, '\\.{5,}', ' '),                '(1-877-FACTSET|www\\.callstreet\\.com|FACTSET ?: ?callstreet'                || '|Copyright © [0-9-]+ FactSet CallStreet, LLC|Corrected Transcript'                || '|Total Pages: [0-9]+|NVIDIA Corp\\. \\(NVDA\\)|Q1 2027 Earnings Call'                || '|[0-9]{1,2}-[A-Z][a-z]{2}-[0-9]{4})', ' '),            '[ \t]{2,}', ' ') AS PARSED_MARKDOWN    FROM RAW_EARNINGS_CALLS)SELECT    r.TICKER,    r.CALL_DATE,    r.FISCAL_PERIOD_LABEL,    r.FILE_NAME,    c.index AS CHUNK_INDEX,    -- 見出しは headers の下にネストしています（## が優先、なければ #）    COALESCE(c.value:headers:header_2::VARCHAR,             c.value:headers:header_1::VARCHAR,             '(不明)') AS SPEAKER,    -- 発言者を「経営陣 / アナリスト / セクション見出し」に分類します    CASE        WHEN COALESCE(c.value:headers:header_2::VARCHAR, c.value:headers:header_1::VARCHAR)             IN ('Colette M. Kress', 'Jen Hsun Huang', 'Jee Hsun Huang', 'Toshiya Hari')            THEN '経営陣'        WHEN COALESCE(c.value:headers:header_2::VARCHAR, c.value:headers:header_1::VARCHAR)             IN ('CORPORATE PARTICIPANTS', 'OTHER PARTICIPANTS', 'MANAGEMENT DISCUSSION SECTION',                 'QUESTION AND ANSWER SECTION', 'Disclaimer', 'NVIDIA Corp. (NVDA)',                 'Q1 2027 Earnings Call')            THEN 'セクション'        ELSE 'アナリスト'    END AS SPEAKER_TYPE,    c.value:chunk::VARCHAR AS CHUNK_TEXTFROM cleaned r,LATERAL FLATTEN(input => SNOWFLAKE.CORTEX.SPLIT_TEXT_MARKDOWN_HEADER(    r.PARSED_MARKDOWN,    OBJECT_CONSTRUCT('#', 'header_1', '##', 'header_2'), -- 見出しレベルとキー名の対応    3000, -- 1チャンクの最大文字数    200 -- チャンク間のオーバーラップ)) cWHERE LENGTH(c.value:chunk::VARCHAR) > 100; -- 短すぎる断片は除外SELECT COUNT(*) AS "チャンク数",       ROUND(AVG(LENGTH(CHUNK_TEXT))) AS "平均文字数",       MAX(LENGTH(CHUNK_TEXT)) AS "最大文字数"FROM GOLD_EARNINGS_CALL_CHUNKS;

In [ ]:
-- 発言者別のチャンク数を確認SELECT SPEAKER_TYPE AS "発言者区分",       SPEAKER AS "発言者",       COUNT(*) AS "チャンク数"FROM GOLD_EARNINGS_CALL_CHUNKSGROUP BY ALLORDER BY "発言者区分", "チャンク数" DESC;

> **💡 確認のポイント**>> CEO（Jen Hsun Huang）と CFO（Colette M. Kress）の発言が最も多く、アナリストの質問も> 発言者ごとに分離できています。>> Part 3 ではこの `SPEAKER` / `SPEAKER_TYPE` を Cortex Search の**属性列**として使い、> 「経営陣の発言だけを検索する」といった絞り込みを実現します。### 1-3. AI_EXTRACT で数値を構造化抽出するチャンク化は「検索できる状態」を作りますが、**数値を集計したい**場合には別のアプローチが必要です。`AI_EXTRACT` は、抽出したい項目を自然言語で指定すると、JSON で値を返してくれます。抽出項目は `OBJECT_CONSTRUCT('キー名', '何を抽出したいかの説明')` で指定します。**説明文がそのまま抽出の指示になる**のがポイントです。

In [ ]:
-- 決算コールから数値ファクトを構造化抽出-- ⚠️ 実行に30秒〜1分程度かかりますCREATE OR REPLACE TABLE GOLD_EARNINGS_CALL_FACTS ASWITH extracted AS (    SELECT        r.TICKER,        r.CALL_DATE,        r.FISCAL_PERIOD_LABEL,        r.PERIOD_END_DATE,        AI_EXTRACT(            file => TO_FILE('@DOC_STAGE', r.FILE_NAME),            responseFormat => OBJECT_CONSTRUCT(                'total_revenue', '当四半期の総売上高（Total revenue）の金額。単位を含めて記載',                'revenue_growth_yoy', '総売上高の前年同期比成長率',                'data_center_revenue', 'データセンター部門の売上高',                'hyperscale_revenue', 'Hyperscale サブマーケットの売上高',                'acie_revenue', 'ACIE サブマーケットの売上高',                'edge_computing_revenue', 'Edge Computing 部門の売上高',                'free_cash_flow', '当四半期のフリーキャッシュフロー',                'total_supply', '当四半期末時点の総サプライ（在庫・購買コミットメント・前払金の合計）',                'next_quarter_revenue_guidance', '次四半期の売上見通し（ガイダンス）の金額',                'next_quarter_gross_margin_guidance', '次四半期の売上総利益率（GAAP）の見通し',                'dividend_change', '四半期配当の変更内容（変更前と変更後の金額）',                'buyback_authorization', '新規に発表された自社株買いの枠の金額'            )        ) AS raw_result    FROM RAW_EARNINGS_CALLS r)SELECT    TICKER,    CALL_DATE,    FISCAL_PERIOD_LABEL,    PERIOD_END_DATE,    raw_result:response.total_revenue::VARCHAR AS TOTAL_REVENUE_TEXT,    raw_result:response.revenue_growth_yoy::VARCHAR AS REVENUE_GROWTH_YOY_TEXT,    raw_result:response.data_center_revenue::VARCHAR AS DATA_CENTER_REVENUE_TEXT,    raw_result:response.hyperscale_revenue::VARCHAR AS HYPERSCALE_REVENUE_TEXT,    raw_result:response.acie_revenue::VARCHAR AS ACIE_REVENUE_TEXT,    raw_result:response.edge_computing_revenue::VARCHAR AS EDGE_COMPUTING_REVENUE_TEXT,    raw_result:response.free_cash_flow::VARCHAR AS FREE_CASH_FLOW_TEXT,    raw_result:response.total_supply::VARCHAR AS TOTAL_SUPPLY_TEXT,    raw_result:response.next_quarter_revenue_guidance::VARCHAR AS NEXT_Q_REVENUE_GUIDANCE_TEXT,    raw_result:response.next_quarter_gross_margin_guidance::VARCHAR AS NEXT_Q_GROSS_MARGIN_GUIDANCE_TEXT,    raw_result:response.dividend_change::VARCHAR AS DIVIDEND_CHANGE_TEXT,    raw_result:response.buyback_authorization::VARCHAR AS BUYBACK_AUTHORIZATION_TEXT,    raw_result AS RAW_EXTRACT_RESULTFROM extracted;SELECT TICKER AS "銘柄",       FISCAL_PERIOD_LABEL AS "決算期",       TOTAL_REVENUE_TEXT AS "総売上",       REVENUE_GROWTH_YOY_TEXT AS "前年同期比",       DATA_CENTER_REVENUE_TEXT AS "データセンター売上",       HYPERSCALE_REVENUE_TEXT AS "Hyperscale売上",       ACIE_REVENUE_TEXT AS "ACIE売上",       EDGE_COMPUTING_REVENUE_TEXT AS "Edge売上",       FREE_CASH_FLOW_TEXT AS "FCF",       NEXT_Q_REVENUE_GUIDANCE_TEXT AS "次Qガイダンス",       NEXT_Q_GROSS_MARGIN_GUIDANCE_TEXT AS "次Q粗利率",       DIVIDEND_CHANGE_TEXT AS "配当変更",       BUYBACK_AUTHORIZATION_TEXT AS "自社株買い枠"FROM GOLD_EARNINGS_CALL_FACTS;

> **⚠️ AI の出力は必ず原文で検証する**>> 抽出結果の **「配当変更」** を見てください。おそらく `0.01 から 0.20` と抽出されています。>> しかし実際の決算コールでは、CFO が `$0.20` と**言い間違え**、その直後の Q&A で CEO が> 「Colette は $0.25 と言うつもりだった」と訂正しています。`AI_EXTRACT` は文書に> 書かれている記述を忠実に抽出するため、**最初に出てきた（誤った）数値を拾いました**。>> これは AI 抽出の限界ではなく、**「文書自体に訂正が含まれている」という現実**です。> 次のセルで原文を確認してみましょう。>> この性質があるため、Part 3 で作る Agent skill には「Cortex Search で取得した記述は> 必ず出典を明記する」というルールを入れています。

In [ ]:
-- 配当に関する発言を原文で確認する-- 「dividend」を含むチャンクを発言順に並べ、訂正のやりとりを確認しますSELECT CHUNK_INDEX AS "順序",       SPEAKER AS "発言者",       SPEAKER_TYPE AS "区分",       -- 発言のうち dividend を含む行を抜き出します       REGEXP_SUBSTR(CHUNK_TEXT, '[^\\n]*dividend[^\\n]*', 1, 1, 'i') AS "配当に関する発言"FROM GOLD_EARNINGS_CALL_CHUNKSWHERE CHUNK_TEXT ILIKE '%dividend%'ORDER BY CHUNK_INDEX;

## 2. AI関数を体験しようここからはニュースデータを題材に、主要な AI 関数を1つずつ試します。まずは小さな `SELECT` で**生の返り値**を確認し、そのあと Gold 層テーブルに合成します。### 2-1. ニュースデータを取り込む`DOC_STAGE` にある `us_company_news.csv`（50件）をテーブルに取り込みます。

In [ ]:
-- ニュース CSV の取り込みCREATE OR REPLACE FILE FORMAT CSV_NEWS_FORMAT    TYPE = 'CSV'    SKIP_HEADER = 1    FIELD_OPTIONALLY_ENCLOSED_BY = '"'    EMPTY_FIELD_AS_NULL = TRUE;CREATE OR REPLACE TABLE RAW_COMPANY_NEWS (    NEWS_ID VARCHAR(10),    PUBLISHED_AT DATE,    TICKER VARCHAR(10),    HEADLINE VARCHAR(500),    BODY VARCHAR(4000),    SOURCE VARCHAR(100),    URL VARCHAR(500));COPY INTO RAW_COMPANY_NEWSFROM @DOC_STAGE/us_company_news.csvFILE_FORMAT = (FORMAT_NAME = CSV_NEWS_FORMAT);SELECT COUNT(*) AS "件数",       COUNT(DISTINCT TICKER) AS "銘柄数",       MIN(PUBLISHED_AT) AS "最古の日付",       MAX(PUBLISHED_AT) AS "最新の日付"FROM RAW_COMPANY_NEWS;

In [ ]:
-- ニュースデータのプレビューSELECT NEWS_ID AS "ID",       PUBLISHED_AT AS "公開日",       TICKER AS "銘柄",       HEADLINE AS "見出し",       LEFT(BODY, 120) AS "本文（冒頭120字）"FROM RAW_COMPANY_NEWSORDER BY PUBLISHED_AT DESCLIMIT 10;

### 2-2. AI_SENTIMENT — センチメント分析テキストの感情極性を判定します。引数はテキスト1つだけです。まずは返り値の JSON 構造を確認しましょう。

In [ ]:
-- AI_SENTIMENT の返り値を確認SELECT TICKER AS "銘柄",       HEADLINE AS "見出し",       AI_SENTIMENT(BODY) AS "返り値（JSON）",       AI_SENTIMENT(BODY):categories[0].sentiment::VARCHAR AS "センチメント"FROM RAW_COMPANY_NEWSWHERE NEWS_ID IN ('N038', 'N004', 'N015', 'N028', 'N020')ORDER BY NEWS_ID;

> **💡 JSON 構造のポイント**>> `AI_SENTIMENT` の返り値は次の形です。>> ```json> {"categories": [{"name": "overall", "sentiment": "positive"}]}> ```>> したがってセンチメントの値を取り出すアクセサは> **`:categories[0].sentiment::VARCHAR`** になります。> 返る値は `positive` / `negative` / `neutral` / `mixed` です。### 2-3. AI_CLASSIFY — テキスト分類テキストを、**自分で定義したラベル**に分類します。ラベルは自由に決められるのが強みです。> **チャレンジ**>> 下のセルのラベル一覧を、自分が「資産運用の現場で使いたい分類」に書き換えて実行してみてください。> 例: `'金利・マクロ'`, `'ESG'`, `'競合動向'`, `'人事・ガバナンス'` など

In [ ]:
-- AI_CLASSIFY でイベント種別に分類-- ラベルは自由に変更できます。自分の分析目的に合わせて書き換えてみてくださいSELECT TICKER AS "銘柄",       HEADLINE AS "見出し",       AI_CLASSIFY(BODY, ['決算・業績', 'ガイダンス', '規制・訴訟', '買収・提携',                          'サプライチェーン', '製品・技術', '株主還元']) AS "返り値（JSON）",       AI_CLASSIFY(BODY, ['決算・業績', 'ガイダンス', '規制・訴訟', '買収・提携',                          'サプライチェーン', '製品・技術', '株主還元']):labels[0]::VARCHAR AS "分類結果"FROM RAW_COMPANY_NEWSWHERE NEWS_ID IN ('N038', 'N013', 'N040', 'N046', 'N017')ORDER BY NEWS_ID;

### 2-4. AI_EXTRACT — 事実の抽出1-3 で決算コールに使ったのと同じ関数を、ニュースにも適用します。> **重要: AI_EXTRACT は「抽出」であって「推論」ではない**>> `AI_EXTRACT` は**文書に書かれている事実**を取り出す関数です。> 「この記事は株価にポジティブか」のような**推論**を求めると `None` が返ってきます。>> | やりたいこと | 使う関数 |> |---|---|> | 文書に書かれている数値・固有名詞を取り出す | `AI_EXTRACT` |> | 感情極性を判定する | `AI_SENTIMENT` |> | 自分で決めたラベルに分類する | `AI_CLASSIFY` |> | 自由な指示で文章を生成・要約する | `AI_COMPLETE` / `AI_AGG` |>> ここでは抽出に向いた項目だけを指定します。

In [ ]:
-- AI_EXTRACT でニュースから事実を抽出SELECT TICKER AS "銘柄",       HEADLINE AS "見出し",       AI_EXTRACT(BODY, OBJECT_CONSTRUCT(           'company_name', '記事の主題となっている企業名',           'key_metric_value', '記事中で最も重要な数値。単位を含めて記載',           'key_metric_name', 'その数値が何を表しているかの説明',           'time_reference', '記事が言及している期間や時点'       )):response AS "抽出結果"FROM RAW_COMPANY_NEWSWHERE NEWS_ID IN ('N038', 'N039', 'N040', 'N043')ORDER BY NEWS_ID;

### 2-5. AI_AGG — 複数行をまとめて要約`AI_SENTIMENT` や `AI_CLASSIFY` は1行ずつ処理しますが、`AI_AGG` は**集約関数**として複数行を横断して1つの出力を生成します。「この銘柄について、直近のニュース全体から何が言えるか」を出したいときに使います。

In [ ]:
-- AI_AGG で銘柄別にニュースを一括要約-- ⚠️ 実行に1分程度かかりますSELECT s.COMPANY_NAME_JA AS "企業名",       n.TICKER AS "銘柄",       COUNT(*) AS "ニュース件数",       AI_AGG(           n.HEADLINE || ' / ' || n.BODY,           'これらは同一企業に関するニュース記事です。投資判断の材料として重要な論点を200文字以内の日本語で要約してください。'           || '前置きや「要約します」といった導入文は書かず、内容から直接始めてください。'           || '好材料と懸念材料の両方に触れてください。投資推奨は書かないでください。'       ) AS "AI要約"FROM RAW_COMPANY_NEWS nJOIN DIM_SECURITY s ON n.TICKER = s.TICKERWHERE n.TICKER IN ('NVDA', 'TSLA', 'AVGO')GROUP BY s.COMPANY_NAME_JA, n.TICKERORDER BY n.TICKER;

## 3. Gold層を構築するここまで個別に試した関数を、1本の CTE チェーンに合成して Gold 層テーブルを作ります。Part 2 の Semantic View と Part 3 の Cortex Search は、このテーブルを参照します。> **⏸️ 休憩の前に実行してください**>> このセルは50件すべてに対して `AI_SENTIMENT` / `AI_CLASSIFY` / `AI_EXTRACT` を適用するため、> **実行に2〜3分かかります**。実行を開始してから休憩に入るのがおすすめです。

In [ ]:
-- Gold層: AI分析済みニューステーブルの構築-- ⚠️ 実行に2〜3分かかりますCREATE OR REPLACE TABLE GOLD_COMPANY_NEWS_ANALYZED ASWITH sentiment_step AS (    -- Step 1: センチメント分析    SELECT n.*,           AI_SENTIMENT(n.BODY):categories[0].sentiment::VARCHAR AS SENTIMENT_EN    FROM RAW_COMPANY_NEWS n),classify_step AS (    -- Step 2: イベント種別の分類    SELECT s.*,           AI_CLASSIFY(s.BODY, ['決算・業績', 'ガイダンス', '規制・訴訟', '買収・提携',                                'サプライチェーン', '製品・技術', '株主還元']):labels[0]::VARCHAR               AS EVENT_CATEGORY    FROM sentiment_step s),extract_step AS (    -- Step 3: 事実の抽出    SELECT c.*,           AI_EXTRACT(c.BODY, OBJECT_CONSTRUCT(               'key_metric_value', '記事中で最も重要な数値。単位を含めて記載',               'key_metric_name', 'その数値が何を表しているかの説明'           )):response AS EXTRACTED    FROM classify_step c)SELECT    e.NEWS_ID,    e.PUBLISHED_AT,    e.TICKER,    d.COMPANY_NAME_JA,    d.SECTOR,    e.HEADLINE,    e.BODY,    e.SOURCE,    e.URL,    -- センチメント（日本語ラベルに変換して業務ユーザーが読める形にします）    e.SENTIMENT_EN,    CASE e.SENTIMENT_EN        WHEN 'positive' THEN 'ポジティブ'        WHEN 'negative' THEN 'ネガティブ'        WHEN 'neutral' THEN 'ニュートラル'        WHEN 'mixed' THEN '混在'        ELSE '判定不能'    END AS SENTIMENT,    e.EVENT_CATEGORY,    e.EXTRACTED:key_metric_value::VARCHAR AS KEY_METRIC_VALUE,    e.EXTRACTED:key_metric_name::VARCHAR AS KEY_METRIC_NAME,    CURRENT_TIMESTAMP() AS PROCESSED_ATFROM extract_step eJOIN DIM_SECURITY d ON e.TICKER = d.TICKER;SELECT COUNT(*) AS "件数", COUNT(DISTINCT SENTIMENT) AS "センチメント種類数",       COUNT(DISTINCT EVENT_CATEGORY) AS "イベント種別数"FROM GOLD_COMPANY_NEWS_ANALYZED;

In [ ]:
-- Gold層テーブルのプレビューSELECT PUBLISHED_AT AS "公開日",       TICKER AS "銘柄",       COMPANY_NAME_JA AS "企業名",       SENTIMENT AS "センチメント",       EVENT_CATEGORY AS "イベント種別",       KEY_METRIC_VALUE AS "主要数値",       KEY_METRIC_NAME AS "数値の意味",       HEADLINE AS "見出し"FROM GOLD_COMPANY_NEWS_ANALYZEDORDER BY PUBLISHED_AT DESCLIMIT 15;

In [ ]:
-- センチメント × イベント種別のクロス集計SELECT EVENT_CATEGORY AS "イベント種別",       COUNT_IF(SENTIMENT = 'ポジティブ') AS "ポジティブ",       COUNT_IF(SENTIMENT = 'ネガティブ') AS "ネガティブ",       COUNT_IF(SENTIMENT = 'ニュートラル') AS "ニュートラル",       COUNT_IF(SENTIMENT = '混在') AS "混在",       COUNT(*) AS "合計"FROM GOLD_COMPANY_NEWS_ANALYZEDGROUP BY ALLORDER BY "合計" DESC;

## 4. 構造化データと突合するここが本パートの核心です。AI 関数の出力は **ただのテーブルの列**です。特別な API も、ベクトルDBも要りません。したがって `setup.sql` で作った株価テーブルや財務諸表テーブルと、**普通の SQL で JOIN できます**。### 4-1. ニュースセンチメント別の翌営業日騰落率「ネガティブなニュースが出た翌営業日、株価は実際に下がっているのか」を検証します。

In [ ]:
-- センチメント別の翌営業日騰落率WITH daily_return AS (    -- 銘柄ごとに前営業日終値との比較で日次騰落率を計算します    SELECT TICKER,           TRADE_DATE,           CLOSE_PRICE,           LAG(CLOSE_PRICE) OVER (PARTITION BY TICKER ORDER BY TRADE_DATE) AS PREV_CLOSE    FROM FACT_STOCK_PRICE_DAILY    WHERE CLOSE_PRICE IS NOT NULL),returns AS (    SELECT TICKER,           TRADE_DATE,           100.0 * (CLOSE_PRICE / PREV_CLOSE - 1) AS DAILY_RETURN_PCT    FROM daily_return    WHERE PREV_CLOSE IS NOT NULL AND PREV_CLOSE > 0),news_next_day AS (    -- ニュース公開日より後の最初の取引日を特定します    SELECT n.NEWS_ID,           n.TICKER,           n.SENTIMENT,           MIN(r.TRADE_DATE) AS NEXT_TRADE_DATE    FROM GOLD_COMPANY_NEWS_ANALYZED n    JOIN returns r      ON r.TICKER = n.TICKER     AND r.TRADE_DATE > n.PUBLISHED_AT    GROUP BY ALL)SELECT nn.SENTIMENT AS "センチメント",       COUNT(*) AS "ニュース件数",       ROUND(AVG(r.DAILY_RETURN_PCT), 2) AS "平均騰落率（%）",       ROUND(MEDIAN(r.DAILY_RETURN_PCT), 2) AS "中央値（%）",       COUNT_IF(r.DAILY_RETURN_PCT > 0) AS "上昇した件数",       COUNT_IF(r.DAILY_RETURN_PCT <= 0) AS "下落した件数"FROM news_next_day nnJOIN returns r  ON r.TICKER = nn.TICKER AND r.TRADE_DATE = nn.NEXT_TRADE_DATEGROUP BY ALLORDER BY "平均騰落率（%）" DESC;

> **💡 結果の読み方**>> センチメントと騰落率にきれいな相関が出ないこともあります。それは失敗ではありません。> 1件のニュースだけで株価が動くわけではなく、市場全体の動きや他の要因が混ざるためです。>> 重要なのは、**「非構造化データから作った列」と「市場データ」を1本の SQL で結合して> 検証できる状態になった**という点です。ここまで来れば、期間を変える・セクターで層別する・> イベント種別で絞るといった分析は、すべて通常の SQL で表現できます。### 4-2. 決算コールの記述と SEC 実績を突合する決算コールで経営陣が語った数値と、SEC 提出書類ベースの実績を並べて確認します。これは実務で必ず必要になる作業です。決算コールでは概数（`$82 billion`）で語られますが、財務諸表には厳密な値（`81,615百万ドル`）が載っています。

In [ ]:
-- 決算コールの記述と SEC 実績の突合SELECT    f.TICKER AS "銘柄",    f.FISCAL_PERIOD_LABEL AS "決算期",    f.CALL_DATE AS "決算発表日",    f.TOTAL_REVENUE_TEXT AS "コールでの記述（総売上）",    TO_VARCHAR(ROUND(m.REVENUE / 1e6), '999,999') || ' 百万ドル' AS "SEC実績（総売上）",    f.REVENUE_GROWTH_YOY_TEXT AS "コールでの記述（前年比）",    -- SEC 実績から前年同期比を自分で計算して照合します    TO_VARCHAR(ROUND(100.0 * (m.REVENUE / prev.REVENUE - 1), 1)) || '%' AS "SEC実績（前年比）",    f.NEXT_Q_REVENUE_GUIDANCE_TEXT AS "次Qガイダンス（コール）"FROM GOLD_EARNINGS_CALL_FACTS fJOIN FACT_FINANCIAL_METRICS m  ON m.TICKER = f.TICKER AND m.PERIOD_END_DATE = f.PERIOD_END_DATE-- 前年同期（約4四半期前）の売上を取得しますJOIN FACT_FINANCIAL_METRICS prev  ON prev.TICKER = f.TICKER AND prev.PERIOD_END_DATE BETWEEN DATEADD('day', -380, f.PERIOD_END_DATE)                              AND DATEADD('day', -350, f.PERIOD_END_DATE);

> **💡 確認のポイント**>> - コールの `$82 billion` と SEC 実績の `81,615 百万ドル` は**概数と厳密値の関係**にあります。>   一致していると断定せず、「概数として整合している」と表現するのが正しい扱いです> - 前年同期比もコールの `85%` と SEC ベースの計算値がほぼ一致します> - セグメント売上の内部整合も確認できます。Hyperscale `$38B` + ACIE `$37B` = Data Center `$75B`、>   さらに Edge Computing `$6.4B` を足すと約 `$81.4B` で、総売上 `$82B` とおおむね整合します

## まとめ### 作成したテーブル| テーブル | 内容 | 件数の目安 ||---|---|---|| `RAW_EARNINGS_CALLS` | 決算コール PDF の Markdown 変換結果 | 1件 || `GOLD_EARNINGS_CALL_CHUNKS` | 発言者単位の検索用チャンク | 約36件 || `GOLD_EARNINGS_CALL_FACTS` | 決算コールから抽出した数値ファクト | 1件 || `RAW_COMPANY_NEWS` | ニュース生データ | 50件 || `GOLD_COMPANY_NEWS_ANALYZED` | AI 分析済みニュース | 50件 |

### 使用した Cortex AI 関数| 関数 | 役割 ||---|---|| `AI_PARSE_DOCUMENT` | PDF を Markdown テキストに変換する || `SPLIT_TEXT_MARKDOWN_HEADER` | Markdown 見出しを境界にチャンク化し、見出しをメタデータとして保持する || `AI_EXTRACT` | 文書に書かれている事実を JSON で抽出する || `AI_SENTIMENT` | 感情極性を判定する || `AI_CLASSIFY` | 自分で定義したラベルに分類する || `AI_AGG` | 複数行を横断して1つの要約を生成する |

### このパートで押さえたポイント1. **PDF はステージに置いたまま SQL だけで構造化できる**。ダウンロードもライブラリ導入も不要2. **決算コールの Markdown 見出しは発言者名になる**。これを使えば発言者単位で検索できる3. **`AI_EXTRACT` は抽出、推論は `AI_SENTIMENT` / `AI_CLASSIFY` の役割**。使い分けが精度を決める4. **AI の出力は原文で検証する**。配当の言い間違いのように、文書自体に訂正が含まれることがある5. **AI 関数の出力はただのテーブルの列**。市場データと普通の SQL で JOIN できる

### 次のステップ`part2_cortex_analyst.ipynb` に進み、これらのテーブルを Semantic View にまとめて自然言語で質問できる状態を作ります。